

**Text embedded per row:** `points_raised` + `conclusion` concatenated into one string.  
Extra columns in the CSV are ignored.

**Output files:**
- `embeddings/embeddings.npy` — matrix of shape (n_rows × 1024)
- `embeddings/records.csv` — original CSV with `embedding_idx` column added

**To add legislation later:** embed chunks with `passage:` prefix and compare against this matrix.

In [ ]:

# !pip install sentence-transformers numpy pandas tqdm

In [2]:
import os
import numpy as np
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
from supabase import create_client
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from numpy.linalg import norm

# Connect to Supabase
load_dotenv()
client = create_client(os.getenv("SUPABASE_URL"), os.getenv("SUPABASE_SERVICE_ROLE_KEY"))

/Users/inge/Desktop/ITU/Thesis/lobby_influence/EU_LOBBYING_INFLUENCE/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load data

In [49]:
# Fetch ALL commission meetings from Supabase (paginated)
PAGE_SIZE = 1000
all_rows = []
offset = 0

while True:
    result = (
        client.table("commission_meetings")
        .select("*")
        .range(offset, offset + PAGE_SIZE - 1)
        .execute()
    )
    rows = result.data or []
    all_rows.extend(rows)
    if len(rows) < PAGE_SIZE:
        break
    offset += PAGE_SIZE

df = pd.DataFrame(all_rows)
print(f"Total rows fetched: {len(df)}")

assert "points_raised" in df.columns, "Missing column: points_raised"
assert "conclusions"    in df.columns, "Missing column: conclusions"

print(f"points_raised non-null: {df['points_raised'].notna().sum()} / {len(df)}")
print(f"conclusions   non-null: {df['conclusions'].notna().sum()} / {len(df)}")

# Filter to rows where points_raised contains "digital omnibus" (case-insensitive)
df = df[df["points_raised"].str.contains("digital euro", case=False, na=False)].reset_index(drop=True)
print(f"\nRows containing 'digital omnibus': {len(df)}")

# Deduplicate on points_raised (keep first occurrence)
before = len(df)
df = df.drop_duplicates(subset=["points_raised"], keep="first").reset_index(drop=True)
print(f"Duplicates removed: {before - len(df)}")
print(f"Rows after dedup:   {len(df)}")

Total rows fetched: 11114
points_raised non-null: 9788 / 11114
conclusions   non-null: 4701 / 11114

Rows containing 'digital omnibus': 50
Duplicates removed: 10
Rows after dedup:   40


## 2. Build text to embed

Concatenate `points_raised` and `conclusion` into one string per row.
Rows where both are empty/N/A get an empty string and will receive a zero vector.

In [43]:
import re

SKIP_VALUES = {"n/a", "na", "none", "nan", ""}  # values treated as empty

def clean_text(text: str) -> str:
    """Remove boilerplate sentences that add noise to embeddings."""
    # 1. Remove "Representatives from (XX) introduced their company" and variants
    text = re.sub(
        r"[^.]*[Rr]epresentatives?\s+from\s+.{1,80}\s+introduced\s+their\s+compan(y|ies)[^.]*\.\s*",
        "", text
    )
    # 2. Remove any sentence containing all of: "Cab", "Dombrovskis", "took", "note" (any order)
    sentences = re.split(r'(?<=[.!?])\s+', text)
    filtered = []
    for sent in sentences:
        sent_lower = sent.lower()
        if all(w in sent_lower for w in ["cab", "dombrovskis", "took", "note"]):
            continue
        filtered.append(sent)
    text = " ".join(filtered)
    # 3. Remove the words "digital" and "omnibus" (case-insensitive)
    text = re.sub(r"\bdigital\b", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\bomnibus\b", "", text, flags=re.IGNORECASE)
    # 4. Collapse whitespace
    text = re.sub(r"\s+", " ", text).strip()
    return text

def build_text(row):
    parts = []
    for col in ["points_raised", "conclusions"]:
        val = str(row[col]).strip() if pd.notna(row[col]) else ""
        if val.lower() not in SKIP_VALUES:
            parts.append(val)
    raw = " ".join(parts)
    return clean_text(raw)

df["_text"] = df.apply(build_text, axis=1)

n_empty = (df["_text"].str.strip() == "").sum()
print(f"Rows with text:    {len(df) - n_empty}")
print(f"Rows with no text: {n_empty}  ← dropping these")

# Drop rows with no text (some may become empty after cleaning)
df = df[df["_text"].str.strip() != ""].reset_index(drop=True)
print(f"Rows after drop:   {len(df)}")

print(f"\nExample concatenated text (row 0):")
print(df["_text"].iloc[0])

Rows with text:    55
Rows with no text: 0  ← dropping these
Rows after drop:   55

Example concatenated text (row 0):
The main point of the meeting was the economic security strategy by Commission, especially for technologies and reducing high-risk dependencies. At the meeting they also discussed the simplification efforts, including to scale up innovations. Google explains investment in the EU and approach on AI developments.


## 3. Load model

`bge-base-en-v1.5` — 110M params, 768-dim embeddings, contrastively trained for sentence similarity.
No prefixes needed for symmetric similarity tasks.

In [44]:
MODEL_NAME = "BAAI/bge-base-en-v1.5"

print(f"Loading {MODEL_NAME} ...")
model = SentenceTransformer(MODEL_NAME)
print(f"Done. Embedding dimension: {model.get_sentence_embedding_dimension()}")

Loading BAAI/bge-base-en-v1.5 ...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5832.02it/s]
BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Done. Embedding dimension: 768


## 4. Embed

Legal-BERT uses raw text (no prefix needed).

In [45]:
BATCH_SIZE = 32  # reduce to 16 if you run out of memory

print(f"Encoding {len(df)} rows in batches of {BATCH_SIZE}...")
print(f"Estimated time: ~{len(df) // BATCH_SIZE // 2 + 1} min on CPU\n")

embeddings = model.encode(
    df["_text"].tolist(),
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True,
)

print(f"\nEmbedding matrix shape: {embeddings.shape}")

Encoding 55 rows in batches of 32...
Estimated time: ~1 min on CPU



Batches: 100%|██████████| 2/2 [00:02<00:00,  1.39s/it]


Embedding matrix shape: (55, 768)


## 5. Save

In [46]:
OUTPUT_DIR = Path("embeddings")
OUTPUT_DIR.mkdir(exist_ok=True)

# Embedding matrix
np.save(OUTPUT_DIR / "validis.npy", embeddings)
print(f"Saved embeddings.npy  shape={embeddings.shape}  ({embeddings.nbytes / 1e6:.1f} MB)")

# CSV with embedding_idx added (== row number, since data is already deduplicated)
df_out = df.drop(columns=["_text"]).copy()
df_out["embedding_idx"] = range(len(df_out))
df_out.to_csv(OUTPUT_DIR / "validis_records.csv", index=False)
print(f"Saved validis_records.csv  ({len(df_out)} rows)")

Saved embeddings.npy  shape=(55, 768)  (0.2 MB)
Saved validis_records.csv  (55 rows)


## 6. Sanity checks

In [47]:
# ── Check 1: similar arguments should score high, unrelated ones low ──────────
print("── Similarity sanity check ──\n")

test_pairs = [
    (
        "The organisation urged the Commission to reconsider CBAM implementation timelines.",
        "Representatives stressed that the carbon border adjustment schedule creates disproportionate burden for SMEs.",
        "SHOULD BE HIGH — same argument, different words"
    ),
    (
        "Representatives from L'Oreal Poland outlined their position on the Urban Wasterwater Treatment Directive. They reported on the assessment carried out by their companies re implementation challenges linked to this Directive. Cab Dombrovskis took note of L'Oreal's position.",
        "The delegation presented their views on fisheries quota allocations in the North Sea.",
        "SHOULD BE LOW — unrelated topics"
    ),
]

for text_a, text_b, label in test_pairs:
    vec_a = model.encode(text_a, normalize_embeddings=True)
    vec_b = model.encode(text_b, normalize_embeddings=True)
    score = float(np.dot(vec_a, vec_b))
    print(f"  {label}")
    print(f"  Score: {score:.3f}\n")

── Similarity sanity check ──

  SHOULD BE HIGH — same argument, different words
  Score: 0.563

  SHOULD BE LOW — unrelated topics
  Score: 0.579



In [48]:
# ── Check 2: nearest neighbours for a sample row ──────────────────────────────
print("── Nearest neighbours for a sample row ──\n")

# Pick first row with a decent-length text
sample_idx = 4
sample_row = df_out.iloc[sample_idx]

print(f"Query (row {sample_idx}):")
print(f"  points_raised: {str(sample_row.get('points_raised', ''))[:400]}")
print(f"  conclusion:    {str(sample_row.get('conclusion', ''))[:150]}\n")

scores = embeddings @ embeddings[sample_idx]
scores[sample_idx] = -1  # exclude self
top5 = np.argsort(scores)[::-1][:5]

print("Top 5 most similar rows:")
for rank, idx in enumerate(top5, 1):
    row = df_out.iloc[idx]
    print(f"  #{rank}  score={scores[idx]:.3f}  row={idx}")
    print(f"       points_raised: {str(row.get('points_raised', ''))[:550]}")
    print()

── Nearest neighbours for a sample row ──

Query (row 4):
  points_raised: Participants discussed the potential content of a digital omnibus and the impact on the competitiveness of the EU economy.
  conclusion:    

Top 5 most similar rows:
  #1  score=0.748  row=52
       points_raised: The representatives of European tech producers presented their views on the upcoming digital omnibus and their policy ideas for improving the European tech ecosystem. The Commission’s representative briefly explained the state of play of the digital omnibus and took note of European tech producer views.

  #2  score=0.725  row=14
       points_raised: Digital Omnibus - Competitiveness of EU market - European defence cooperation

  #3  score=0.725  row=50
       points_raised: - The representatives of Schneider Electric and the Commission had a discussion on the current simplification process regarding the EU digital omnibus package. Schneider Electric addressed the importance of establishing rules tha

## 7. How to load and use in future notebooks

Copy this block — you never need to re-embed unless you change the model.

In [ ]:
# import numpy as np
# import pandas as pd
# from sentence_transformers import SentenceTransformer
#
# embeddings = np.load("embeddings/embeddings.npy")
# df         = pd.read_csv("embeddings/records.csv")
#
# # Similarity between row i and all others:
# scores = embeddings @ embeddings[i]
#
# # Add legislation later — no re-embedding of meetings needed:
# model   = SentenceTransformer("intfloat/multilingual-e5-large")
# law_vec = model.encode("passage: Article 4 shall apply to...", normalize_embeddings=True)
# scores  = embeddings @ law_vec
# top5    = np.argsort(scores)[::-1][:5]

print("See comments above.")

## 7b. Parse and embed legislation PDF

Chunks the legislation into recitals and Article 1 paragraphs,
then embeds each chunk with a `passage:` prefix.

Output:
- `legislation_embeddings/legislation_chunks.json`
- `legislation_embeddings/legislation_embeddings.npy`

In [29]:
import re
import json
import numpy as np
import pdfplumber
from pathlib import Path

PDF_PATH = "COM_COM_2025_0836_EN.pdf"  # ← update to your filename

# ── 1. Extract full text ─────────────────────────────────────────────────────
def extract_text(pdf_path):
    text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            t = page.extract_text()
            if t:
                text += t + "\n"
    return text

# ── 2. Find section boundaries ───────────────────────────────────────────────
def find_bounds(text):
    bounds = {}
    for key, pattern in [
        ("recitals_start", r'Whereas:\n'),
        ("recitals_end",   r'HAVE ADOPTED THIS REGULATION'),
        ("article1_start", r'\nArticle 1\nAmendments'),
        ("article2_start", r'\nArticle 2\n'),
    ]:
        m = re.search(pattern, text)
        bounds[key] = m.start() if m else None
    return bounds

# ── 3. Parse recitals ────────────────────────────────────────────────────────
def parse_recitals(text, start, end):
    section = text[start:end]
    chunks, matches = [], list(re.compile(r'\((\d+)\)\s+').finditer(section))
    for i, match in enumerate(matches):
        num = int(match.group(1))
        content = section[match.end(): matches[i+1].start() if i+1 < len(matches) else len(section)]
        content = re.sub(r'\s+', ' ', content).strip()
        if len(content) < 50:
            continue
        chunks.append({"chunk_id": f"recital_{num}", "type": "recital",
                        "number": num, "text": content})
    return chunks

# ── 4. Parse Article 1 paragraphs ────────────────────────────────────────────
def parse_article1(text, start, end):
    section = text[start:end]
    chunks, matches = [], list(re.compile(r'\n\((\d+)\)\s+').finditer(section))
    for i, match in enumerate(matches):
        num = int(match.group(1))
        content = section[match.end(): matches[i+1].start() if i+1 < len(matches) else len(section)]
        content = re.sub(r'\s+', ' ', content).strip()
        if len(content) < 30:
            continue
        ref = re.search(r'(?:in |Article |amend(?:ing|s)?\s+)Article\s+([\d\w\(\)]+)',
                         content[:150], re.IGNORECASE)
        chunks.append({"chunk_id": f"article1_para_{num}", "type": "article1_paragraph",
                        "number": num, "ai_act_ref": ref.group(0).strip() if ref else "",
                        "text": content})
    return chunks

# ── 5. Run chunking ──────────────────────────────────────────────────────────
print(f"Extracting text from {PDF_PATH} ...")
leg_text   = extract_text(PDF_PATH)
bounds     = find_bounds(leg_text)
recitals   = parse_recitals(leg_text, bounds['recitals_start'], bounds['recitals_end'])
articles   = parse_article1(leg_text, bounds['article1_start'], bounds['article2_start'])
leg_chunks = recitals + articles

for i, c in enumerate(leg_chunks):
    c['embedding_idx'] = i

print(f"  Recitals:            {len(recitals)}")
print(f"  Article 1 paragraphs: {len(articles)}")
print(f"  Total chunks:         {len(leg_chunks)}")


Extracting text from COM_COM_2025_0836_EN.pdf ...
  Recitals:            46
  Article 1 paragraphs: 33
  Total chunks:         79


In [30]:
# ── 6. Embed legislation chunks (no prefix for Legal-BERT) ────────────────────
texts = [c["text"] for c in leg_chunks]

print(f"Embedding {len(texts)} legislation chunks...")
leg_embeddings = model.encode(
    texts,
    batch_size=16,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True,
)
print(f"Shape: {leg_embeddings.shape}")

# ── 7. Save ───────────────────────────────────────────────────────────────────
LEG_DIR = Path("legislation_embeddings")
LEG_DIR.mkdir(exist_ok=True)

np.save(LEG_DIR / "legislation_embeddings.npy", leg_embeddings)
with open(LEG_DIR / "legislation_chunks.json", "w", encoding="utf-8") as f:
    json.dump(leg_chunks, f, ensure_ascii=False, indent=2)

print(f"Saved legislation_embeddings.npy  shape={leg_embeddings.shape}")
print(f"Saved legislation_chunks.json     ({len(leg_chunks)} chunks)")

# Quick preview
print("\nFirst 5 chunk IDs:")
for c in leg_chunks[:5]:
    print(f"  {c['chunk_id']:<25} {c['text'][:80]}...")

Embedding 79 legislation chunks...


Batches: 100%|██████████| 5/5 [00:05<00:00,  1.06s/it]

Shape: (79, 768)
Saved legislation_embeddings.npy  shape=(79, 768)
Saved legislation_chunks.json     (79 chunks)

First 5 chunk IDs:
  recital_1                 Regulation (EU) 2024/1689 of the European Parliament and of the Council3 lays do...
  recital_2                 The experience gathered in implementing the parts of Regulation (EU) 2024/1689 t...
  recital_3                 Consequently, targeted amendments to Regulation (EU) 2024/1689 are necessary to ...
  recital_4                 Enterprises outgrowing the micro, small and medium-sized enterprises (‘SME’) def...
  recital_5                 Article 4 of Regulation (EU) 2024/1689 currently imposes an obligation on all pr...


In [32]:
import numpy as np
import pandas as pd
import json

# Load everything
meet_emb = np.load("embeddings/validis.npy")
meet_df  = pd.read_csv("embeddings/validis_records.csv")



with open("legislation_embeddings/legislation_chunks.json") as f:
    leg_chunks = json.load(f)
leg_emb = np.load("legislation_embeddings/legislation_embeddings.npy")
print("meeting emb norm:", np.linalg.norm(meet_emb[0]))
print("legislation emb norm:", np.linalg.norm(leg_emb[0]))
# Build a quick lookup: chunk_id → index
chunk_lookup = {c["chunk_id"]: i for i, c in enumerate(leg_chunks)}

# Query function — given a chunk ID, show the chunk and top 5 most similar meeting rows
def top5(chunk_id):
    idx = chunk_lookup[chunk_id]
    vec = leg_emb[idx]
    
    chunk = leg_chunks[idx]
    print(f"CHUNK: {chunk_id}")
    print(f"TYPE:  {chunk['type']}")
    if chunk.get("ai_act_ref"):
        print(f"AMENDS: {chunk['ai_act_ref']}")
    print(f"TEXT:  {chunk['text'][:800]}")
    print()
    
    scores = meet_emb @ vec
    top    = np.argsort(scores)[::-1][:5]
    
    for rank, i in enumerate(top, 1):
        row = meet_df.iloc[i]
        print(f"  #{rank}  score={scores[i]:.3f}")
        print(f"  org:  {str(row.get('interest_representatives', ''))[:70]}")
        print(f"  date: {row.get('date', '')}")
        print(f"  text: {str(row.get('points_raised', ''))[:800]}")
        print()

# ── List all available chunk IDs ──────────────────────────────────────────────
def list_chunks():
    for c in leg_chunks:
        label = c.get("ai_act_ref") or c.get("section") or ""
        print(f"  {c['chunk_id']:<30}  {label[:60]}")

# ── Usage ─────────────────────────────────────────────────────────────────────
list_chunks()

meeting emb norm: 0.99999994
legislation emb norm: 0.99999994
  recital_1                       
  recital_2                       
  recital_3                       
  recital_4                       
  recital_5                       
  recital_6                       
  recital_7                       
  recital_8                       
  recital_3                       
  recital_9                       
  recital_3                       
  recital_2                       
  recital_3                       
  recital_10                      
  recital_1                       
  recital_11                      
  recital_12                      
  recital_13                      
  recital_14                      
  recital_1                       
  recital_9                       
  recital_15                      
  recital_16                      
  recital_1                       
  recital_17                      
  recital_18                      
  recital_19                

In [34]:

top5("recital_8")       # bias detection / personal data


CHUNK: recital_8
TYPE:  recital
TEXT:  With a view to ensuring the smooth application and consistency of Regulation (EU) 2024/1689, amendments should be made to it. A technical correction to Article 43(3), first subparagraph, of Regulation (EU) 2024/1689 should be added to align the conformity assessment requirements with the requirements of providers of high-risk AI systems in Article 16 of that Regulation. Moreover, it should be clarified that where a provider of a high-risk AI system is subject to the conformity assessment procedure under Union harmonisation legislation listed in Section A of Annex I to Regulation (EU) 2024/1689, and the conformity assessment extends to compliance of the quality management system of that Regulation and of such Union harmonisation legislation, the provider should be able to include aspects re

  #1  score=0.930
  org:  
  date: 
  text: The representatives from Cosmetics Europe communicated they are following the simplification legislation with much 